# Leveäniemi Process-Water Forecasting

This notebook rebuilds a machine-learning surrogate for the consultant's 2013 process-water workbook and then runs 2026-2030 forecasts from new ore-input assumptions.

**Important context:** the GK/GL values inside the consultant workbook are not treated as measured current/future truth. They are consultant-derived scenario/proxy values. LK, Leveaniemi-Kiruna, was not directly considered by the consultant, so this notebook infers an LK proxy from the available Leveaniemi and Kiruna leaching-rate structure. The model learns from the consultant's recurrence logic; future forecasts should use user-supplied new ore production/leaching inputs when available, or otherwise be reported as sensitivity analysis.

The workbook column `J` is treated as an inflow/pit-pump concentration. The recurrence output used for training is the consultant's GM result column `AI`, while `AF` is retained as the storage/state concentration that is fed forward. If your thesis definition of the target should instead be a different column, change `TARGET_COLUMN` in the configuration cell.

## Dependencies

Run this once in the notebook kernel if the imports below fail:

```python
%pip install pandas openpyxl scikit-learn matplotlib xlsxwriter
```

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import math
import warnings

REQUIRED_MODULES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
}
missing = [package for module, package in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Missing required packages: " + ", ".join(missing) +
        ". Install them with `%pip install pandas openpyxl scikit-learn matplotlib xlsxwriter`."
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# -----------------------------
# Project configuration
# -----------------------------
WORKBOOK_NAME = "Leveaniemi_data.xlsx"

def find_workbook(filename: str = WORKBOOK_NAME) -> Path:
    candidates = [Path(filename), Path.cwd() / filename]
    candidates.extend(parent / filename for parent in Path.cwd().resolve().parents)
    candidates.extend([
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
        Path("/home/ojaami/dev/water-quality-prediction") / filename,
    ])

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate.resolve()

    try:
        from google.colab import files
    except Exception:
        files = None

    if files is not None:
        print(f"{filename} was not found in the Colab runtime. Upload the Excel file now.")
        uploaded = files.upload()
        exact_path = Path.cwd() / filename
        if exact_path.exists():
            return exact_path.resolve()
        excel_uploads = [Path.cwd() / name for name in uploaded if name.lower().endswith((".xlsx", ".xlsm", ".xls"))]
        if excel_uploads:
            print(f"Using uploaded workbook: {excel_uploads[0].name}")
            return excel_uploads[0].resolve()

    searched = "\n".join(str(candidate) for candidate in unique_candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Current notebook working directory is {Path.cwd()}. "
        f"If you are using Colab, upload {filename} to /content or mount Drive. Searched:\n{searched}"
    )

WORKBOOK_PATH = find_workbook()
print(f"Using workbook: {WORKBOOK_PATH}")
PROCESS_WATER_SHEET = "Process water"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "gm_output_conc"       # consultant's GM recurrence result in column AI
STATE_COLUMN = "prev_storage_conc"     # concentration fed into the next recurrence step
TRAIN_END_YEAR = 2025.5
FORECAST_START_YEAR = 2026.0
FORECAST_END_YEAR = 2030.0
FORECAST_YEARS = np.round(np.arange(FORECAST_START_YEAR, FORECAST_END_YEAR + 0.01, 0.5), 1)

RANDOM_STATE = 42
N_MONTE_CARLO = 500        # thesis requirement: >= 200
LEACH_PERTURBATION = 0.15  # +/-15% uniform perturbation on leaching load/rate proxy

# Optional concentration limits. Fill these if the permit/project thresholds are known.
# The units must match PARAM_BLOCKS: Cu/Ni in ug/l, NH4/Cl in mg/l.
ELEMENT_LIMITS = {
    "Cu": None,
    "NH4": None,
    "Cl": None,
    "Ni": None,
}

# IMPORTANT: GK/GL columns in the workbook are consultant scenario/proxy values,
# not measured current/future ore-plan truth. LK is not in the workbook directly;
# it is inferred from the available Leveaniemi and Kiruna rate structure.
# Current operations can use time-varying LK ratios, such as 50/50 or 40/60.
# These fallback values are useful for sensitivity only.
# If real 2026-2030 ore inputs are available, replace None with a DataFrame/list of
# rows containing: parameter, year, production_mton, process_leach.
# Optional columns: ore_frac_gm, ore_frac_gk, ore_frac_gl, ore_frac_lk,
# lk_leveaniemi_frac, lk_kiruna_frac.
NEW_ORE_INPUTS = None

# Optional proxy schedule for varying ore mixes over time. Fill this when the mine
# uses different LK ratios in different years or seasons. Example:
# TIME_VARYING_ORE_MIX = pd.DataFrame([
#     {"year": 2026.0, "lk": 1.0, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
#     {"year": 2026.5, "lk": 1.0, "lk_leveaniemi": 0.40, "lk_kiruna": 0.60},
# ])
TIME_VARYING_ORE_MIX = None

# Fallback sensitivity scenarios when NEW_ORE_INPUTS is None.
# Fractions are normalized by row, so values only need to be proportional.
ORE_COMPONENTS = ["gm", "gk", "gl", "lk"]
LK_INTERNAL_COMPONENTS = ["lk_leveaniemi", "lk_kiruna"]
LK_DEFAULT_LEVEANIEMI_FRACTION = 0.50
LK_DEFAULT_KIRUNA_FRACTION = 0.50
ORE_MIX_SCENARIOS = {
    "GK100": {"gm": 0.0, "gk": 1.0, "gl": 0.0, "lk": 0.0},
    "GL100": {"gm": 0.0, "gk": 0.0, "gl": 1.0, "lk": 0.0},
    "LK50_50": {"gm": 0.0, "gk": 0.0, "gl": 0.0, "lk": 1.0, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
    "LK40_60": {"gm": 0.0, "gk": 0.0, "gl": 0.0, "lk": 1.0, "lk_leveaniemi": 0.40, "lk_kiruna": 0.60},
    "LK60_40": {"gm": 0.0, "gk": 0.0, "gl": 0.0, "lk": 1.0, "lk_leveaniemi": 0.60, "lk_kiruna": 0.40},
    "GK60_GL40": {"gm": 0.0, "gk": 0.60, "gl": 0.40, "lk": 0.0},
    "GK50_LK50": {"gm": 0.0, "gk": 0.50, "gl": 0.0, "lk": 0.50, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
    "GL50_LK50": {"gm": 0.0, "gk": 0.0, "gl": 0.50, "lk": 0.50, "lk_leveaniemi": 0.50, "lk_kiruna": 0.50},
}
SELECTED_ORE_MIX_SCENARIO = TIME_VARYING_ORE_MIX if TIME_VARYING_ORE_MIX is not None else "LK50_50"
SELECTED_ORE_MIX_LABEL = "TIME_VARYING_ORE_MIX" if TIME_VARYING_ORE_MIX is not None else SELECTED_ORE_MIX_SCENARIO
FORECAST_INPUT_MODE = (
    "user_supplied_new_ore_inputs" if NEW_ORE_INPUTS is not None
    else "time_varying_proxy_ore_mix" if TIME_VARYING_ORE_MIX is not None
    else "consultant_proxy_sensitivity"
)

PARAM_BLOCKS = {
    "Cu":  {"unit": "ug/l", "data_start_excel_row": 43,  "max_rows": 35},
    "NH4": {"unit": "mg/l", "data_start_excel_row": 82,  "max_rows": 35},
    "Cl":  {"unit": "mg/l", "data_start_excel_row": 122, "max_rows": 35},
    "Ni":  {"unit": "ug/l", "data_start_excel_row": 161, "max_rows": 35},
}

## Workbook Extraction

The extraction uses explicit zero-indexed Excel columns so the logic remains visible. The column names below mirror the consultant's sheet: production/leaching alternatives, pit-pump input, water-balance flows, storage volume, storage state, and GM recurrence result.

In [ ]:
COLUMN_MAP = {
    "year": 0,
    "tailings_load": 1,
    "roll_leach_rate": 2,
    "production_mton_gm": 3,
    "process_leach_gm": 4,
    "production_mton_gk": 5,
    "process_leach_gk": 6,
    "process_leach_gl": 7,
    "pit_pump_volume": 8,
    "pit_pump_conc": 9,
    "ditch_sw_flow": 10,
    "ditch_sw_conc": 11,
    "ditch_se_flow": 12,
    "ditch_se_conc": 13,
    "gruvberget_flow": 14,
    "gruvberget_conc": 15,
    "surface_water_flow": 16,
    "surface_water_conc": 17,
    "process_loss_flow": 18,
    "process_loss_conc": 19,
    "dams_from_process_flow": 20,
    "dams_from_process_conc": 21,
    "p2_process_flow": 22,
    "p2_process_conc": 23,
    "tailings_storage_flow": 24,
    "tailings_storage_conc": 25,
    "discharge_flow": 26,
    "discharge_conc": 27,
    "leakage_flow": 28,
    "leakage_conc": 29,
    "storage_volume": 30,
    "storage_conc_state_col_af": 31,
    "losses": 32,
    "gains": 33,
    "gm_output_conc": 34,
}

GAIN_FLOW_COLUMNS = [
    "pit_pump_volume",
    "ditch_sw_flow",
    "ditch_se_flow",
    "gruvberget_flow",
    "surface_water_flow",
]
LOSS_FLOW_COLUMNS = [
    "process_loss_flow",
    "tailings_storage_flow",
    "discharge_flow",
    "leakage_flow",
]
LOAD_PAIRS = [
    ("pit_pump_volume", "pit_pump_conc"),
    ("ditch_sw_flow", "ditch_sw_conc"),
    ("ditch_se_flow", "ditch_se_conc"),
    ("gruvberget_flow", "gruvberget_conc"),
    ("surface_water_flow", "surface_water_conc"),
    ("process_loss_flow", "process_loss_conc"),
    ("tailings_storage_flow", "tailings_storage_conc"),
    ("discharge_flow", "discharge_conc"),
    ("leakage_flow", "leakage_conc"),
]

def _numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _clean_proxy_rate(series: pd.Series) -> pd.Series:
    cleaned = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    median = cleaned.median(skipna=True)
    fill_value = median if np.isfinite(median) else 0.0
    return cleaned.fillna(fill_value)


def add_lk_proxy_inputs(extracted: pd.DataFrame) -> pd.DataFrame:
    """Infer an LK, Leveaniemi-Kiruna, process-leaching proxy from workbook rates.

    The consultant workbook has GK and GL proxy columns, but no direct LK column.
    This derives Leveaniemi and Kiruna rates from those formulas. The actual
    LK ratio can then be set as 50/50, 40/60, or any time-varying schedule.
    Replace this with NEW_ORE_INPUTS if measured LK data exists.
    """
    out = extracted.copy()
    gm_rate = out["roll_leach_rate"]
    prod_gm = out["production_mton_gm"].replace(0, np.nan)
    prod_gk = out["production_mton_gk"].replace(0, np.nan)

    kiruna_rate = (out["process_leach_gk"] - gm_rate * 0.4 * out["production_mton_gm"]) / (0.6 * prod_gk)
    leveaniemi_rate = ((out["process_leach_gl"] / prod_gm) - gm_rate * 0.4) / 0.6

    out["kiruna_leach_rate_proxy"] = _clean_proxy_rate(kiruna_rate)
    out["leveaniemi_leach_rate_proxy"] = _clean_proxy_rate(leveaniemi_rate)
    out["production_mton_lk"] = out["production_mton_gm"]
    out["lk_leveaniemi_frac"] = LK_DEFAULT_LEVEANIEMI_FRACTION
    out["lk_kiruna_frac"] = LK_DEFAULT_KIRUNA_FRACTION
    out["process_leach_lk"] = (
        out["lk_leveaniemi_frac"] * out["leveaniemi_leach_rate_proxy"] +
        out["lk_kiruna_frac"] * out["kiruna_leach_rate_proxy"]
    ) * out["production_mton_lk"]
    return out


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create compact mass-balance features that help the RF learn the recurrence."""
    out = df.copy()

    out[GAIN_FLOW_COLUMNS + LOSS_FLOW_COLUMNS] = out[GAIN_FLOW_COLUMNS + LOSS_FLOW_COLUMNS].fillna(0.0)
    out["gross_gain_flow"] = out[GAIN_FLOW_COLUMNS].sum(axis=1)
    out["gross_loss_flow"] = out[LOSS_FLOW_COLUMNS].sum(axis=1)
    out["net_flow"] = out["gross_gain_flow"] - out["gross_loss_flow"]

    load_terms = []
    for flow_col, conc_col in LOAD_PAIRS:
        term_name = f"{flow_col}_load"
        out[term_name] = out[flow_col].fillna(0.0) * out[conc_col].fillna(0.0)
        load_terms.append(term_name)
    out["known_water_load_proxy"] = out[load_terms].sum(axis=1)

    out["production_mton"] = out["production_mton"].replace(0, np.nan)
    out["storage_volume"] = out["storage_volume"].replace(0, np.nan)
    out["leach_per_mton"] = out["process_leach"] / out["production_mton"]
    out["leach_per_storage_volume"] = out["process_leach"] / out["storage_volume"]
    out["storage_mass_proxy"] = out["storage_volume"] * out[STATE_COLUMN]
    out["pump_conc_x_volume"] = out["pit_pump_volume"].fillna(0.0) * out["pit_pump_conc"].fillna(0.0)
    return out


def extract_parameter_block(process_water: pd.DataFrame, parameter: str, spec: dict) -> pd.DataFrame:
    start = spec["data_start_excel_row"] - 1
    raw = process_water.iloc[start:start + spec["max_rows"], :].copy()

    extracted = pd.DataFrame({name: _numeric(raw.iloc[:, idx]) for name, idx in COLUMN_MAP.items()})
    extracted["source_excel_row"] = raw.index + 1
    extracted = extracted[extracted["year"].between(2014.0, 2030.5, inclusive="both")].copy()
    extracted = extracted.sort_values("year").reset_index(drop=True)

    extracted["parameter"] = parameter
    extracted["unit"] = spec["unit"]
    extracted["calendar_year"] = np.floor(extracted["year"]).astype(int)
    extracted["half_year"] = np.isclose(extracted["year"] % 1, 0.5).astype(int)
    extracted["year_index"] = extracted["year"] - extracted["year"].min()
    extracted["tailings_load"] = extracted["tailings_load"].fillna(0.0)

    # AI is the consultant's GM result. If the terminal row lacks AI, keep AF as a fallback only.
    extracted["gm_output_conc"] = extracted["gm_output_conc"].fillna(extracted["storage_conc_state_col_af"])

    # The recurrence state for row t is the previous modelled output. Seed the first row from AF.
    extracted[STATE_COLUMN] = extracted["gm_output_conc"].shift(1)
    if not extracted.empty:
        extracted.loc[0, STATE_COLUMN] = extracted.loc[0, "storage_conc_state_col_af"]
    extracted["is_initial_row"] = False
    if not extracted.empty:
        extracted.loc[0, "is_initial_row"] = True

    # Historical training uses the original GM-heavy assumption.
    extracted["ore_frac_gm"] = 1.0
    extracted["ore_frac_gk"] = 0.0
    extracted["ore_frac_gl"] = 0.0
    extracted["ore_frac_lk"] = 0.0
    extracted = add_lk_proxy_inputs(extracted)
    extracted["production_mton"] = extracted["production_mton_gm"]
    extracted["process_leach"] = extracted["process_leach_gm"]

    return add_derived_features(extracted)


process_water = pd.read_excel(WORKBOOK_PATH, sheet_name=PROCESS_WATER_SHEET, header=None, engine="openpyxl")
blocks = [extract_parameter_block(process_water, parameter, spec) for parameter, spec in PARAM_BLOCKS.items()]
model_data = pd.concat(blocks, ignore_index=True)

summary = model_data.groupby("parameter").agg(
    rows=("year", "count"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    first_output=(TARGET_COLUMN, "first"),
    last_output=(TARGET_COLUMN, "last"),
)
display(summary)
display(model_data.loc[:, ["parameter", "year", "half_year", "production_mton_gm", "process_leach_gm", "process_leach_gk", "process_leach_gl", "process_leach_lk", STATE_COLUMN, TARGET_COLUMN]].head(12))

## Ore-Input Substitution

The model learns from the consultant's calculated recurrence rows. For forecasting, use `NEW_ORE_INPUTS` when real new ore production/leaching assumptions are available. If `NEW_ORE_INPUTS = None`, the notebook uses workbook GK/GL proxy columns and an inferred LK proxy as clearly labelled sensitivity scenarios, not as observed truth.

In [ ]:
def normalize_mix(mix: dict) -> dict:
    values = {component: float(mix.get(component, 0.0)) for component in ORE_COMPONENTS}
    total = sum(values.values())
    if total <= 0:
        raise ValueError("Ore mix fractions must sum to a positive value.")
    return {component: value / total for component, value in values.items()}


def normalize_lk_internal_mix(mix: dict) -> dict:
    values = {
        "lk_leveaniemi": float(mix.get("lk_leveaniemi", LK_DEFAULT_LEVEANIEMI_FRACTION)),
        "lk_kiruna": float(mix.get("lk_kiruna", LK_DEFAULT_KIRUNA_FRACTION)),
    }
    total = sum(values.values())
    if total <= 0:
        return {"lk_leveaniemi": LK_DEFAULT_LEVEANIEMI_FRACTION, "lk_kiruna": LK_DEFAULT_KIRUNA_FRACTION}
    return {component: value / total for component, value in values.items()}


def scenario_to_table(scenario: str | dict | pd.DataFrame, years=FORECAST_YEARS) -> pd.DataFrame:
    if isinstance(scenario, str):
        if scenario not in ORE_MIX_SCENARIOS:
            raise KeyError(f"Unknown ore-mix scenario: {scenario}")
        scenario_def = ORE_MIX_SCENARIOS[scenario]
        mix = normalize_mix(scenario_def)
        lk_mix = normalize_lk_internal_mix(scenario_def)
        return pd.DataFrame({"year": years, **mix, **lk_mix})

    if isinstance(scenario, dict):
        mix = normalize_mix(scenario)
        lk_mix = normalize_lk_internal_mix(scenario)
        return pd.DataFrame({"year": years, **mix, **lk_mix})

    table = scenario.copy()
    if "year" not in table.columns:
        raise ValueError("Scenario table is missing required column: year")
    for component in ORE_COMPONENTS:
        if component not in table.columns:
            table[component] = 0.0
    for component in LK_INTERNAL_COMPONENTS:
        if component not in table.columns:
            table[component] = LK_DEFAULT_LEVEANIEMI_FRACTION if component == "lk_leveaniemi" else LK_DEFAULT_KIRUNA_FRACTION
    fractions = table[ORE_COMPONENTS].astype(float)
    totals = fractions.sum(axis=1).replace(0, np.nan)
    table[ORE_COMPONENTS] = fractions.div(totals, axis=0)
    lk_fractions = table[LK_INTERNAL_COMPONENTS].astype(float)
    lk_totals = lk_fractions.sum(axis=1).replace(0, np.nan)
    table[LK_INTERNAL_COMPONENTS] = lk_fractions.div(lk_totals, axis=0).fillna({"lk_leveaniemi": LK_DEFAULT_LEVEANIEMI_FRACTION, "lk_kiruna": LK_DEFAULT_KIRUNA_FRACTION})
    return table[["year", *ORE_COMPONENTS, *LK_INTERNAL_COMPONENTS]]


def apply_new_ore_inputs(df: pd.DataFrame, ore_inputs) -> pd.DataFrame | None:
    """Apply measured/user-supplied forecast inputs instead of consultant proxy columns."""
    if ore_inputs is None:
        return None

    table = pd.DataFrame(ore_inputs).copy()
    required = {"parameter", "year", "production_mton", "process_leach"}
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f"NEW_ORE_INPUTS is missing required columns: {sorted(missing)}")

    ore_fraction_columns = [f"ore_frac_{component}" for component in ORE_COMPONENTS]
    lk_internal_columns = [f"{component}_frac" for component in LK_INTERNAL_COMPONENTS]
    for col in ore_fraction_columns:
        if col not in table.columns:
            table[col] = np.nan
    for col in lk_internal_columns:
        if col not in table.columns:
            table[col] = np.nan

    table = table[["parameter", "year", "production_mton", "process_leach", *ore_fraction_columns, *lk_internal_columns]].copy()
    table["year"] = pd.to_numeric(table["year"], errors="coerce")
    for col in ["production_mton", "process_leach", *ore_fraction_columns, *lk_internal_columns]:
        table[col] = pd.to_numeric(table[col], errors="coerce")
    table = table.rename(columns={"production_mton": "production_mton_new", "process_leach": "process_leach_new", **{col: f"{col}_new" for col in [*ore_fraction_columns, *lk_internal_columns]}})

    out = df.drop(columns=[*ore_fraction_columns, *lk_internal_columns], errors="ignore").copy()
    out = out.merge(table, on=["parameter", "year"], how="left")
    missing_rows = out[out["process_leach_new"].isna() | out["production_mton_new"].isna()][["parameter", "year"]]
    if not missing_rows.empty:
        raise ValueError("NEW_ORE_INPUTS does not cover all forecast rows:\n" + missing_rows.to_string(index=False))
    out["production_mton"] = out["production_mton_new"]
    out["process_leach"] = out["process_leach_new"]
    for col in ore_fraction_columns:
        out[col] = out[f"{col}_new"].fillna(0.0)
    for col in lk_internal_columns:
        default = LK_DEFAULT_LEVEANIEMI_FRACTION if col == "lk_leveaniemi_frac" else LK_DEFAULT_KIRUNA_FRACTION
        out[col] = out[f"{col}_new"].fillna(default)
    out = out.drop(columns=["production_mton_new", "process_leach_new", *[f"{col}_new" for col in [*ore_fraction_columns, *lk_internal_columns]]])
    return add_derived_features(out)


def apply_ore_mix(df: pd.DataFrame, scenario: str | dict | pd.DataFrame, ore_inputs=NEW_ORE_INPUTS) -> pd.DataFrame:
    out = df.copy()
    user_input_df = apply_new_ore_inputs(out, ore_inputs)
    if user_input_df is not None:
        return user_input_df

    mix_table = scenario_to_table(scenario, years=out["year"].to_numpy())
    ore_fraction_columns = [f"ore_frac_{component}" for component in ORE_COMPONENTS]
    lk_internal_columns = [f"{component}_frac" for component in LK_INTERNAL_COMPONENTS]
    out = out.drop(columns=[*ore_fraction_columns, *lk_internal_columns], errors="ignore")
    rename_map = {component: f"ore_frac_{component}" for component in ORE_COMPONENTS}
    rename_map.update({component: f"{component}_frac" for component in LK_INTERNAL_COMPONENTS})
    out = out.merge(mix_table.rename(columns=rename_map), on="year", how="left")
    out[ore_fraction_columns] = out[ore_fraction_columns].fillna({"ore_frac_gm": 1.0, "ore_frac_gk": 0.0, "ore_frac_gl": 0.0, "ore_frac_lk": 0.0})
    out[lk_internal_columns] = out[lk_internal_columns].fillna({"lk_leveaniemi_frac": LK_DEFAULT_LEVEANIEMI_FRACTION, "lk_kiruna_frac": LK_DEFAULT_KIRUNA_FRACTION})
    lk_total = out["lk_leveaniemi_frac"] + out["lk_kiruna_frac"]
    out["lk_leveaniemi_frac"] = np.where(lk_total > 0, out["lk_leveaniemi_frac"] / lk_total, LK_DEFAULT_LEVEANIEMI_FRACTION)
    out["lk_kiruna_frac"] = np.where(lk_total > 0, out["lk_kiruna_frac"] / lk_total, LK_DEFAULT_KIRUNA_FRACTION)

    # Consultant-proxy fallback only: workbook GK/GL and inferred LK values are not observed new ore inputs.
    # GL uses the same production schedule column as the workbook's GL leaching-load formula.
    production_mton_gl_proxy = out["production_mton_gm"]
    out["process_leach_lk_dynamic"] = (
        out["lk_leveaniemi_frac"] * out["leveaniemi_leach_rate_proxy"] +
        out["lk_kiruna_frac"] * out["kiruna_leach_rate_proxy"]
    ) * out["production_mton_lk"]
    out["production_mton"] = (
        out["ore_frac_gm"] * out["production_mton_gm"] +
        out["ore_frac_gk"] * out["production_mton_gk"] +
        out["ore_frac_gl"] * production_mton_gl_proxy +
        out["ore_frac_lk"] * out["production_mton_lk"]
    )
    out["process_leach"] = (
        out["ore_frac_gm"] * out["process_leach_gm"] +
        out["ore_frac_gk"] * out["process_leach_gk"] +
        out["ore_frac_gl"] * out["process_leach_gl"] +
        out["ore_frac_lk"] * out["process_leach_lk_dynamic"]
    )
    return add_derived_features(out)


selected_mix_table = scenario_to_table(SELECTED_ORE_MIX_SCENARIO)
selected_mix_table["input_mode"] = FORECAST_INPUT_MODE
display(selected_mix_table.head())

## Model Training and Diagnostics

The Random Forest is trained separately for each contaminant. Features are imputed and standardized within each parameter block so chloride's mg/kg-scale leaching values do not dominate the other blocks. If a full model has weak cross-validated R², the notebook also evaluates separate winter/summer models and uses them if they improve the diagnostic score.

In [ ]:
FEATURE_COLUMNS = [
    "year_index",
    "calendar_year",
    "half_year",
    "tailings_load",
    "ore_frac_gm",
    "ore_frac_gk",
    "ore_frac_gl",
    "ore_frac_lk",
    "production_mton",
    "process_leach",
    "leach_per_mton",
    "leach_per_storage_volume",
    "pit_pump_volume",
    "pit_pump_conc",
    "pump_conc_x_volume",
    "gross_gain_flow",
    "gross_loss_flow",
    "net_flow",
    "known_water_load_proxy",
    "storage_volume",
    STATE_COLUMN,
    "storage_mass_proxy",
    "losses",
    "gains",
]


def make_model(seed: int = RANDOM_STATE) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("rf", RandomForestRegressor(
                n_estimators=800,
                min_samples_leaf=1,
                max_features="sqrt",
                bootstrap=True,
                random_state=seed,
                n_jobs=-1,
            )),
        ]
    )


def cv_predict(model: Pipeline, X: pd.DataFrame, y: pd.Series, max_splits: int = 5) -> np.ndarray:
    n = len(y)
    if n < 4:
        return np.full(n, np.nan)
    splits = min(max_splits, n)
    cv = KFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_predict(model, X, y, cv=cv, n_jobs=None)


def evaluate_half_year_models(train: pd.DataFrame) -> tuple[float, dict[int, Pipeline], pd.Series]:
    preds = pd.Series(index=train.index, dtype=float)
    models: dict[int, Pipeline] = {}
    for half_value in [0, 1]:
        sub = train[train["half_year"] == half_value]
        if len(sub) < 4:
            continue
        X_sub = sub[FEATURE_COLUMNS]
        y_sub = sub[TARGET_COLUMN]
        preds.loc[sub.index] = cv_predict(make_model(RANDOM_STATE + half_value + 10), X_sub, y_sub, max_splits=3)
        model = make_model(RANDOM_STATE + half_value + 100)
        model.fit(X_sub, y_sub)
        models[half_value] = model
    if preds.notna().all():
        return r2_score(train[TARGET_COLUMN], preds.loc[train.index]), models, preds
    return np.nan, models, preds


def train_parameter_models(data: pd.DataFrame):
    model_packs = {}
    diagnostic_rows = []
    cv_rows = []

    for parameter, param_df in data.groupby("parameter", sort=False):
        train = param_df[
            (~param_df["is_initial_row"]) &
            (param_df["year"] <= TRAIN_END_YEAR) &
            param_df[TARGET_COLUMN].notna()
        ].copy()
        train = train.dropna(subset=[TARGET_COLUMN])
        X = train[FEATURE_COLUMNS]
        y = train[TARGET_COLUMN]

        full_model_for_cv = make_model(RANDOM_STATE)
        full_cv_pred = cv_predict(full_model_for_cv, X, y, max_splits=5)
        full_r2 = r2_score(y, full_cv_pred) if np.isfinite(full_cv_pred).all() else np.nan
        full_mae = mean_absolute_error(y, full_cv_pred) if np.isfinite(full_cv_pred).all() else np.nan

        half_r2, half_models, half_preds = evaluate_half_year_models(train)
        use_half_models = bool(np.isfinite(half_r2) and (not np.isfinite(full_r2) or half_r2 > full_r2 + 0.03) and full_r2 < 0.70)

        final_model = make_model(RANDOM_STATE)
        final_model.fit(X, y)

        model_packs[parameter] = {
            "parameter": parameter,
            "unit": train["unit"].iloc[0],
            "full_model": final_model,
            "half_models": half_models,
            "use_half_models": use_half_models,
            "training_rows": len(train),
            "full_cv_r2": full_r2,
            "full_cv_mae": full_mae,
            "half_cv_r2": half_r2,
        }

        diagnostic_rows.append({
            "parameter": parameter,
            "unit": train["unit"].iloc[0],
            "training_rows": len(train),
            "full_cv_r2": full_r2,
            "full_cv_mae": full_mae,
            "half_year_cv_r2": half_r2,
            "forecast_model": "separate half-year RF" if use_half_models else "single RF with half_year feature",
        })

        for idx, row in train.iterrows():
            cv_rows.append({
                "parameter": parameter,
                "year": row["year"],
                "observed": row[TARGET_COLUMN],
                "cv_pred_single_rf": full_cv_pred[list(train.index).index(idx)] if np.isfinite(full_cv_pred).all() else np.nan,
                "cv_pred_half_year_rf": half_preds.loc[idx] if idx in half_preds.index else np.nan,
            })

    diagnostics = pd.DataFrame(diagnostic_rows).sort_values("parameter")
    cv_predictions = pd.DataFrame(cv_rows).sort_values(["parameter", "year"])
    return model_packs, diagnostics, cv_predictions


model_packs, diagnostics, cv_predictions = train_parameter_models(model_data)
display(diagnostics)

## Sequential Forecast and Monte Carlo Uncertainty

For each Monte Carlo run, the forecast leaching input is perturbed by ±15%. The predicted concentration from each half-year row becomes the storage-state feature for the next row. If `NEW_ORE_INPUTS` is still `None`, these forecasts are sensitivity scenarios based on consultant-proxy GK/GL columns and inferred LK proxy values, not final new-ore predictions.

In [ ]:
def predict_from_pack(model_pack: dict, feature_row: pd.DataFrame) -> float:
    if model_pack["use_half_models"]:
        half_value = int(feature_row["half_year"].iloc[0])
        model = model_pack["half_models"].get(half_value, model_pack["full_model"])
    else:
        model = model_pack["full_model"]
    prediction = float(model.predict(feature_row[FEATURE_COLUMNS])[0])
    return max(prediction, 0.0)


def sequential_forecast(
    parameter: str,
    data: pd.DataFrame,
    model_pack: dict,
    scenario: str | dict | pd.DataFrame = SELECTED_ORE_MIX_SCENARIO,
    ore_inputs=NEW_ORE_INPUTS,
    n_runs: int = N_MONTE_CARLO,
    leach_perturbation: float = LEACH_PERTURBATION,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    param_df = data[data["parameter"] == parameter].sort_values("year").copy()
    forecast_template = param_df[param_df["year"].isin(FORECAST_YEARS)].copy()
    forecast_template = apply_ore_mix(forecast_template, scenario, ore_inputs=ore_inputs).sort_values("year")
    input_mode = "user_supplied_new_ore_inputs" if ore_inputs is not None else "consultant_proxy_sensitivity"

    historical_state = param_df[param_df["year"] <= TRAIN_END_YEAR].sort_values("year")
    start_state = float(historical_state[TARGET_COLUMN].dropna().iloc[-1])

    rng = np.random.default_rng(random_state + sum(ord(ch) for ch in parameter))
    records = []
    for run in range(n_runs):
        state = start_state
        for _, template_row in forecast_template.iterrows():
            row = template_row.copy()
            row[STATE_COLUMN] = state
            row["process_leach"] = row["process_leach"] * rng.uniform(1.0 - leach_perturbation, 1.0 + leach_perturbation)
            feature_row = add_derived_features(pd.DataFrame([row]))
            prediction = predict_from_pack(model_pack, feature_row)
            state = prediction
            records.append({
                "run": run,
                "input_mode": input_mode,
                "parameter": parameter,
                "unit": model_pack["unit"],
                "year": float(row["year"]),
                "calendar_year": int(row["calendar_year"]),
                "half_year": int(row["half_year"]),
                "ore_frac_gm": float(row["ore_frac_gm"]),
                "ore_frac_gk": float(row["ore_frac_gk"]),
                "ore_frac_gl": float(row["ore_frac_gl"]),
                "ore_frac_lk": float(row["ore_frac_lk"]),
                "lk_leveaniemi_frac": float(row.get("lk_leveaniemi_frac", LK_DEFAULT_LEVEANIEMI_FRACTION)),
                "lk_kiruna_frac": float(row.get("lk_kiruna_frac", LK_DEFAULT_KIRUNA_FRACTION)),
                "prediction": prediction,
            })

    return pd.DataFrame(records)


def summarize_forecast(simulations: pd.DataFrame) -> pd.DataFrame:
    group_columns = ["parameter", "unit", "year", "calendar_year", "half_year", "ore_frac_gm", "ore_frac_gk", "ore_frac_gl", "ore_frac_lk", "lk_leveaniemi_frac", "lk_kiruna_frac"]
    if "input_mode" in simulations.columns:
        group_columns.insert(2, "input_mode")
    summary = (
        simulations
        .groupby(group_columns, dropna=False)["prediction"]
        .quantile([0.10, 0.50, 0.90])
        .unstack()
        .rename(columns={0.10: "p10", 0.50: "p50", 0.90: "p90"})
        .reset_index()
        .sort_values(["parameter", "year"])
    )
    return summary


forecast_simulations = pd.concat(
    [sequential_forecast(parameter, model_data, model_packs[parameter]) for parameter in PARAM_BLOCKS],
    ignore_index=True,
)
forecast_summary = summarize_forecast(forecast_simulations)
annual_forecast_summary = forecast_summary[np.isclose(forecast_summary["year"] % 1, 0.0)].copy()

display(forecast_summary.head(12))
display(annual_forecast_summary)

## Proxy Ore-Combination Sensitivity

When measured new ore inputs are unavailable, this grid shows how forecasts respond under consultant-proxy GK, GL, and inferred LK ore-combination scenarios. Treat this as sensitivity analysis, not as evidence that the workbook proxy values are actual future conditions.

In [ ]:
def run_proxy_sensitivity(scenarios: dict | None = None, runs_per_mix: int = 200) -> pd.DataFrame:
    frames = []
    scenarios = scenarios or ORE_MIX_SCENARIOS
    for scenario_index, (scenario_name, scenario) in enumerate(scenarios.items()):
        sims = pd.concat(
            [
                sequential_forecast(
                    parameter,
                    model_data,
                    model_packs[parameter],
                    scenario=scenario,
                    ore_inputs=None,
                    n_runs=runs_per_mix,
                    random_state=RANDOM_STATE + scenario_index * 1000,
                )
                for parameter in PARAM_BLOCKS
            ],
            ignore_index=True,
        )
        summary = summarize_forecast(sims)
        normalized = normalize_mix(scenario)
        summary["sensitivity_scenario"] = scenario_name
        for component in ORE_COMPONENTS:
            summary[f"sensitivity_{component}_fraction"] = normalized[component]
        lk_normalized = normalize_lk_internal_mix(scenario)
        summary["sensitivity_lk_leveaniemi_fraction"] = lk_normalized["lk_leveaniemi"]
        summary["sensitivity_lk_kiruna_fraction"] = lk_normalized["lk_kiruna"]
        frames.append(summary)
    return pd.concat(frames, ignore_index=True)


sensitivity_summary = run_proxy_sensitivity()
sensitivity_annual = sensitivity_summary[np.isclose(sensitivity_summary["year"] % 1, 0.0)].copy()

display(sensitivity_annual.head(20))

## Thesis Figure

In [ ]:
def plot_forecast_panels(data: pd.DataFrame, forecast: pd.DataFrame, diagnostics: pd.DataFrame) -> Path:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axes = axes.ravel()

    for ax, parameter in zip(axes, PARAM_BLOCKS):
        hist = data[(data["parameter"] == parameter) & (data["year"] <= TRAIN_END_YEAR)].sort_values("year")
        fc = forecast[forecast["parameter"] == parameter].sort_values("year")
        unit = PARAM_BLOCKS[parameter]["unit"]
        diag = diagnostics[diagnostics["parameter"] == parameter].iloc[0]
        r2 = diag["half_year_cv_r2"] if diag["forecast_model"] == "separate half-year RF" else diag["full_cv_r2"]

        ax.plot(
            hist["year"], hist[TARGET_COLUMN],
            color="#1f2937", marker="o", linewidth=1.8, markersize=4,
            label="Consultant GM model"
        )
        ax.fill_between(
            fc["year"].to_numpy(), fc["p10"].to_numpy(), fc["p90"].to_numpy(),
            color="#6aaed6", alpha=0.28, label="P10-P90"
        )
        ax.plot(fc["year"], fc["p50"], color="#0b6e99", marker="o", linewidth=2.0, markersize=4, label="Forecast P50")
        limit = ELEMENT_LIMITS.get(parameter)
        if limit is not None and np.isfinite(float(limit)):
            ax.axhline(float(limit), color="#b91c1c", linestyle=":", linewidth=1.6, label="Limit")
        ax.axvline(TRAIN_END_YEAR, color="#9ca3af", linestyle="--", linewidth=1.2)
        ax.set_title(f"{parameter} ({unit}) | CV R²={r2:.2f}")
        ax.set_ylabel(f"Concentration ({unit})")
        ax.grid(True, alpha=0.25)

    for ax in axes[-2:]:
        ax.set_xlabel("Year")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
    input_label = "user-supplied new ore inputs" if NEW_ORE_INPUTS is not None else f"proxy ore mix: {SELECTED_ORE_MIX_LABEL}"
    fig.suptitle(f"Leveäniemi Process-Water Forecast ({input_label}) with ±15% Leaching Uncertainty", y=0.98, fontsize=15)
    fig.tight_layout(rect=(0, 0.05, 1, 0.95))

    out_path = OUTPUT_DIR / "leveaniemi_forecast_bands.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    return out_path


figure_path = plot_forecast_panels(model_data, forecast_summary, diagnostics)
figure_path

## Excel Export

In [ ]:
forecast_path = OUTPUT_DIR / "leveaniemi_forecast_values.xlsx"
element_limits = pd.DataFrame([
    {"parameter": parameter, "unit": PARAM_BLOCKS[parameter]["unit"], "limit": limit}
    for parameter, limit in ELEMENT_LIMITS.items()
])
forecast_input_note = pd.DataFrame([
    {
        "input_mode": FORECAST_INPUT_MODE,
        "selected_proxy_scenario": SELECTED_ORE_MIX_LABEL if NEW_ORE_INPUTS is None else None,
        "note": "Workbook GK/GL values are consultant-derived proxy/sensitivity inputs, and LK is inferred from Leveaniemi and Kiruna proxy rates because the consultant did not include LK directly. LK ratios can vary by year/season through TIME_VARYING_ORE_MIX. These are not measured current/future ore-plan truth unless confirmed by the mine. Provide NEW_ORE_INPUTS for final new-ore forecasts.",
    }
])


with pd.ExcelWriter(forecast_path) as writer:
    diagnostics.to_excel(writer, sheet_name="Diagnostics", index=False)
    cv_predictions.to_excel(writer, sheet_name="CV_predictions", index=False)
    forecast_summary.to_excel(writer, sheet_name="Forecast_half_year", index=False)
    annual_forecast_summary.to_excel(writer, sheet_name="Forecast_annual", index=False)
    sensitivity_annual.to_excel(writer, sheet_name="Sensitivity_annual", index=False)
    selected_mix_table.to_excel(writer, sheet_name="Selected_proxy_mix", index=False)
    element_limits.to_excel(writer, sheet_name="Element_limits", index=False)
    forecast_input_note.to_excel(writer, sheet_name="Forecast_input_note", index=False)

print(f"Saved figure: {figure_path}")
print(f"Saved forecast workbook: {forecast_path}")

## Optional: Join Daily Monitoring Data

If daily monitoring data is available later, load it here, aggregate it by year or season, and plot it against `forecast_summary`. Keep it separate from training unless the thesis question changes from reproducing the consultant's recurrence to calibrating against observations.

In [ ]:
# Example scaffold for later monitoring-data comparison:
# monitoring_path = Path("daily_monitoring.xlsx")
# monitoring = pd.read_excel(monitoring_path)
# monitoring["date"] = pd.to_datetime(monitoring["date"])
# monitoring["year"] = monitoring["date"].dt.year + np.where(monitoring["date"].dt.month >= 6, 0.5, 0.0)
# seasonal_monitoring = monitoring.groupby(["parameter", "year"], as_index=False)["concentration"].median()
# display(seasonal_monitoring.head())